# SL-1 - Apprentissage Logique : CBH Search et Version Space

**Navigation** : [Index](./README.md) | [EBL & RBL >>](SL-2-KnowledgeBasedLearning.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. Representer des hypotheses comme des formules logiques (conjonctions de predicats)
2. Definir la notion de consistance d'une hypothese (faux positifs / faux negatifs)
3. Implementer l'algorithme **Current-Best-Hypothesis** (CBH) search avec generalisation et specialisation
4. Implementer l'algorithme **Version Space / Candidate Elimination** avec G-set et S-set
5. Comprendre les limites de l'apprentissage par espace de versions (bruit, disjonctions)

### Prerequis
- Python 3.10+
- Connaissances basiques en logique propositionnelle et predicats
- Notions de generalisation / specialisation en apprentissage

### Duree estimee : 50 minutes

### References
- Russell & Norvig, *Artificial Intelligence: A Modern Approach*, 3e ed., Chapitre 19.1
- Tom Mitchell, *Machine Learning*, Chapitre 2 (Concept Learning)

---

## 1. Le probleme restaurant (AIMA)

L'exemple classique de l'AIMA pour l'apprentissage logique est le **probleme du restaurant** : etant donne un ensemble d'experiences de restaurant (decrites par des attributs), apprendre la regle **WillWait** qui predit si on attendra ou non pour une table.

### Attributs du domaine

| Attribut | Description | Valeurs possibles |
|----------|-------------|-------------------|
| `Alternate` | Autre restaurant a cote ? | Yes, No |
| `Bar` | Bar/salle d'attente ? | Yes, No |
| `Fri/Sat` | Vendredi ou samedi ? | Yes, No |
| `Hungry` | Faim ? | Yes, No |
| `Patrons` | Frequentation | None, Some, Full |
| `Price` | Prix | $, $$, $$$ |
| `Raining` | Pleut-il ? | Yes, No |
| `Reservation` | Reservation ? | Yes, No |
| `Type` | Type de restaurant | French, Italian, Thai, Burger |
| `WaitEstimate` | Temps d'attente estime | 0-10, 10-30, 30-60, >60 |

### Hypotheses comme conjonctions

Une hypothese est une **conjonction de contraintes** sur les attributs. Chaque contrainte peut etre :
- Une valeur specifique : `Patrons = Full`
- Un joker (`?`) : n'importe quelle valeur (attribut ignore)
- Une valeur `None` : aucun exemple ne doit avoir cette valeur

L'hypothese la plus generale est `(?, ?, ?, ?, ?, ?, ?, ?, ?, ?)` — elle accepte tout.
L'hypothese la plus specifique est `(None, None, None, None, None, None, None, None, None, None)` — elle refuse tout.

In [1]:
# Definition du domaine : attributs et leurs valeurs possibles
ATTRIBUTES = [
    "Alternate",    # Yes, No
    "Bar",          # Yes, No
    "Fri/Sat",      # Yes, No
    "Hungry",       # Yes, No
    "Patrons",      # None, Some, Full
    "Price",        # $, $$, $$$
    "Raining",      # Yes, No
    "Reservation",  # Yes, No
    "Type",         # French, Italian, Thai, Burger
    "WaitEstimate", # 0-10, 10-30, 30-60, >60
]

ATTRIBUTE_VALUES = {
    "Alternate":   ["Yes", "No"],
    "Bar":         ["Yes", "No"],
    "Fri/Sat":     ["Yes", "No"],
    "Hungry":      ["Yes", "No"],
    "Patrons":     ["None", "Some", "Full"],
    "Price":       ["$", "$$", "$$$"],
    "Raining":     ["Yes", "No"],
    "Reservation": ["Yes", "No"],
    "Type":        ["French", "Italian", "Thai", "Burger"],
    "WaitEstimate": ["0-10", "10-30", "30-60", ">60"],
}

# Les 12 exemples du restaurant (AIMA Table 19.1)
RAW_EXAMPLES = [
    # (Alternate, Bar, Fri/Sat, Hungry, Patrons, Price, Raining, Reservation, Type, WaitEstimate, WillWait)
    ("Yes", "No",  "No",  "Yes", "Some", "$$$", "No",  "Yes",  "French", "0-10",  True),
    ("Yes", "No",  "No",  "Yes", "Full", "$",   "No",  "No",   "Thai",   "30-60", True),
    ("No",  "Yes", "No",  "No",  "Some", "$",   "No",  "No",   "Burger", "0-10",  False),
    ("Yes", "No",  "Yes", "Yes", "Full", "$",   "Yes", "No",   "Thai",   "10-30", True),
    ("Yes", "No",  "Yes", "No",  "Full", "$$$", "No",  "Yes",  "French", ">60",   False),
    ("No",  "Yes", "No",  "Yes", "Some", "$$",  "Yes", "Yes",  "Italian","0-10",  True),
    ("No",  "Yes", "No",  "No",  "None", "$",   "Yes", "No",  "Burger", "0-10",  False),
    ("No",  "No",  "No",  "Yes", "Some", "$$",  "Yes", "Yes",  "Thai",   "0-10",  True),
    ("No",  "Yes", "Yes", "No",  "Full", "$",   "Yes", "No",  "Burger", "10-30", False),
    ("Yes", "Yes", "Yes", "Yes", "Full", "$$$", "No",  "Yes",  "Italian","10-30", False),
    ("No",  "No",  "No",  "No",  "None", "$",   "No",  "No",   "Thai",   "0-10",  False),
    ("Yes", "Yes", "Yes", "Yes", "Full", "$",   "No",  "No",   "Burger", "30-60", True),
]

def parse_example(raw: tuple) -> dict:
    """Convertit un tuple brut en dictionnaire avec attributs + label."""
    attrs = {ATTRIBUTES[i]: raw[i] for i in range(len(ATTRIBUTES))}
    attrs["WillWait"] = raw[len(ATTRIBUTES)]
    return attrs

EXAMPLES = [parse_example(ex) for ex in RAW_EXAMPLES]

print(f"Domaine : {len(ATTRIBUTES)} attributs")
print(f"Exemples : {len(EXAMPLES)} ({sum(1 for e in EXAMPLES if e['WillWait'])} positifs, "
      f"{sum(1 for e in EXAMPLES if not e['WillWait'])} negatifs)")

Domaine : 10 attributs
Exemples : 12 (6 positifs, 6 negatifs)


### Interpretation : Domaine du restaurant

| Element | Valeur | Signification |
|---------|--------|---------------|
| Attributs | 10 | Description complete de chaque experience restaurant |
| Exemples | 12 (6 positifs, 6 negatifs) | Donnees d'apprentissage |
| Tache | Classifier WillWait | Predire s'il faut attendre une table |

Les exemples positifs (WillWait=True) sont des situations ou on a attendu avec satisfaction, les negatifs (WillWait=False) des situations ou l'attente n'en valait pas la peine.

> **Note** : En comptant le joker `?` et la valeur interdite `None` par attribut, l'espace **syntaxique** des hypotheses conjonctives compte `prod(|valeurs_i| + 2)` combinaisons, soit `4^6 * 5^2 * 6^2 = 3 686 400` (6 attributs binaires, 2 attributs a 3 valeurs, 2 attributs a 4 valeurs). L'espace **semantique** (hypotheses distinctes en extension) est plus petit : toute hypothese contenant un `None` rejette tous les exemples, elles sont donc toutes equivalentes a une unique hypothese vide, d'ou `prod(|valeurs_i| + 1) + 1 = 3^6 * 4^2 * 5^2 + 1 = 291 601`. L'espace des hypotheses est borne mais immense au regard des 12 exemples.


---

## 2. Representation des hypotheses et consistance

Une **hypothese** est un tuple de contraintes, une par attribut. Trois valeurs possibles par position :
- `"?"` (joker) : accepte n'importe quelle valeur
- Une valeur specifique (ex `"Full"`) : accepte uniquement cette valeur
- `None` : valeur speciale impossible (utilisee pour le S-set initial)

### Consistance

Une hypothese est **consistante** avec un exemple si :
- L'exemple est **positif** et l'hypothese le couvre (pas de faux negatif)
- L'exemple est **negatif** et l'hypothese ne le couvre pas (pas de faux positif)

In [2]:
from typing import Optional

# Type alias pour les hypotheses
Hypothesis = tuple[Optional[str], ...]  # Chaque element : "?" | valeur | None

# Hypotheses extremes
G0: Hypothesis = ("?",) * len(ATTRIBUTES)   # La plus generale
S0: Hypothesis = (None,) * len(ATTRIBUTES)  # La plus specifique


def covers(h: Hypothesis, example: dict) -> bool:
    """Une hypothese couvre un exemple si chaque contrainte est satisfaite."""
    for i, attr in enumerate(ATTRIBUTES):
        if h[i] is None:
            # Contrainte impossible -> ne couvre rien
            return False
        if h[i] != "?" and h[i] != example[attr]:
            return False
    return True


def is_consistent(h: Hypothesis, examples: list[dict]) -> bool:
    """Verifie la consistance d'une hypothese avec tous les exemples."""
    for ex in examples:
        h_covers = covers(h, ex)
        if ex["WillWait"] and not h_covers:
            return False  # Faux negatif
        if not ex["WillWait"] and h_covers:
            return False  # Faux positif
    return True


def count_errors(h: Hypothesis, examples: list[dict]) -> tuple[int, int]:
    """Compte les faux positifs et faux negatifs."""
    fp = fn = 0
    for ex in examples:
        h_covers = covers(h, ex)
        if not ex["WillWait"] and h_covers:
            fp += 1
        if ex["WillWait"] and not h_covers:
            fn += 1
    return fp, fn


# Test sur les hypotheses extremes
print(f"G0 (plus generale) : {G0}")
print(f"  Couvre exemple 1 : {covers(G0, EXAMPLES[0])}")
print(f"  Faux positifs / Faux negatifs : {count_errors(G0, EXAMPLES)}")
print(f"  Consistante : {is_consistent(G0, EXAMPLES)}")
print()
print(f"S0 (plus specifique) : {S0}")
print(f"  Couvre exemple 1 : {covers(S0, EXAMPLES[0])}")
print(f"  Faux positifs / Faux negatifs : {count_errors(S0, EXAMPLES)}")
print(f"  Consistante : {is_consistent(S0, EXAMPLES)}")

G0 (plus generale) : ('?', '?', '?', '?', '?', '?', '?', '?', '?', '?')
  Couvre exemple 1 : True
  Faux positifs / Faux negatifs : (6, 0)
  Consistante : False

S0 (plus specifique) : (None, None, None, None, None, None, None, None, None, None)
  Couvre exemple 1 : False
  Faux positifs / Faux negatifs : (0, 6)
  Consistante : False


### Interpretation : Consistance

| Hypothese | Couvre | Faux positifs | Faux negatifs | Consistante |
|-----------|--------|---------------|---------------|-------------|
| G0 (tout `?`) | Tout | 6 (tous les negatifs) | 0 | Non |
| S0 (tout `None`) | Rien | 0 | 6 (tous les positifs) | Non |

**G0** est trop generale : elle couvre tous les exemples, y compris les negatifs (6 faux positifs).
**S0** est trop specifique : elle ne couvre aucun exemple, meme pas les positifs (6 faux negatifs).

L'objectif des algorithmes CBH et Version Space est de trouver une hypothese **entre les deux** qui minimise ces erreurs.

---

## 3. Generalisation et specialisation

Les deux operations fondamentales pour manipuler les hypotheses :

- **Generaliser** : remplacer une contrainte specifique par `?` (accepter plus d'exemples)
- **Specialiser** : remplacer un `?` par une valeur specifique (accepter moins d'exemples)

### Hierarchie de generalisation

```
(?, ?, ?, ...)           <- plus generale
  |                       (generalisation)
(Full, ?, ?, ...)
  |
(Full, $, ?, ...)        <- plus specifique
```

Une hypothese h1 est **plus generale que** h2 (notation h1 >= h2) si h2 couvre un sous-ensemble des exemples couverts par h1.

In [3]:
def is_more_general_than(h1: Hypothesis, h2: Hypothesis) -> bool:
    """h1 est-elle plus generale que h2 ? (h1 >= h2)"""
    for i in range(len(ATTRIBUTES)):
        # h1[i] doit etre "?" ou egal a h2[i] pour chaque position
        if h1[i] != "?" and h1[i] != h2[i]:
            return False
    return True


def is_more_specific_than(h1: Hypothesis, h2: Hypothesis) -> bool:
    """h1 est-elle plus specifique que h2 ? (h1 <= h2)"""
    return is_more_general_than(h2, h1)


def generalize_for(h: Hypothesis, example: dict) -> Hypothesis:
    """Generalise h minimalement pour couvrir l'exemple positif."""
    result = list(h)
    for i, attr in enumerate(ATTRIBUTES):
        if result[i] is None or result[i] == example[attr]:
            result[i] = example[attr]
        elif result[i] != "?" and result[i] != example[attr]:
            result[i] = "?"
    return tuple(result)


def specialize_against(h: Hypothesis, example: dict) -> list[Hypothesis]:
    """Specialise h pour ne PAS couvrir l'exemple negatif.
    
    Retourne toutes les specialisations minimales possibles.
    """
    specializations = []
    for i, attr in enumerate(ATTRIBUTES):
        if h[i] == "?":
            # Remplacer "?" par chaque valeur possible SAUF celle de l'exemple negatif
            for val in ATTRIBUTE_VALUES[attr]:
                if val != example[attr]:
                    new_h = list(h)
                    new_h[i] = val
                    specializations.append(tuple(new_h))
    return specializations


# Demonstration
h_test = ("Yes", "No", "No", "Yes", "Full", "$", "No", "No", "Thai", "30-60")
print(f"Hypothese initiale : {h_test}")
print(f"Couvre exemple 2 (positif) : {covers(h_test, EXAMPLES[1])}")
print()

# Generaliser pour couvrir l'exemple 1 (qui differe sur Patrons et Price)
h_gen = generalize_for(h_test, EXAMPLES[0])
print(f"Apres generalisation pour ex1 : {h_gen}")
print(f"Couvre maintenant ex1 : {covers(h_gen, EXAMPLES[0])}")
print(f"Couvre toujours ex2 : {covers(h_gen, EXAMPLES[1])}")
print()

# Specialiser contre l'exemple 3 (negatif)
h_gen2 = generalize_for(S0, EXAMPLES[0])
h_gen2 = generalize_for(h_gen2, EXAMPLES[1])
print(f"Hypothese apres 2 positifs : {h_gen2}")
specs = specialize_against(h_gen2, EXAMPLES[2])
print(f"Specialisations contre ex3 (negatif) : {len(specs)}")
for s in specs[:5]:
    print(f"  {s}")
if len(specs) > 5:
    print(f"  ... et {len(specs) - 5} autres")

Hypothese initiale : ('Yes', 'No', 'No', 'Yes', 'Full', '$', 'No', 'No', 'Thai', '30-60')
Couvre exemple 2 (positif) : True

Apres generalisation pour ex1 : ('Yes', 'No', 'No', 'Yes', '?', '?', 'No', '?', '?', '?')
Couvre maintenant ex1 : True
Couvre toujours ex2 : True

Hypothese apres 2 positifs : ('Yes', 'No', 'No', 'Yes', '?', '?', 'No', '?', '?', '?')
Specialisations contre ex3 (negatif) : 11
  ('Yes', 'No', 'No', 'Yes', 'None', '?', 'No', '?', '?', '?')
  ('Yes', 'No', 'No', 'Yes', 'Full', '?', 'No', '?', '?', '?')
  ('Yes', 'No', 'No', 'Yes', '?', '$$', 'No', '?', '?', '?')
  ('Yes', 'No', 'No', 'Yes', '?', '$$$', 'No', '?', '?', '?')
  ('Yes', 'No', 'No', 'Yes', '?', '?', 'No', 'Yes', '?', '?')
  ... et 6 autres


### Interpretation : Generalisation et specialisation

| Operation | Effet | Quand l'utiliser |
|-----------|-------|------------------|
| **Generaliser** | Remplace les contraintes differentes par `?` | Quand un faux negatif est detecte (exemple positif non couvert) |
| **Specialiser** | Remplace un `?` par une valeur specifique | Quand un faux positif est detecte (exemple negatif couvert) |

**Observations** :
- La generalisation est **deterministe** : une seule generalisation minimale
- La specialisation est **non deterministe** : plusieurs specialisations possibles (une par valeur d'attribut eligible)
- Le nombre de specialisations croit rapidement : pour un attribut binaire, 1 specialisation; pour un attribut a 4 valeurs, 3 specialisations

> **Point cle** : Cette asymetrie entre generalisation (unique) et specialisation (multiple) est la raison pour laquelle CBH peut revenir en arriere (backtracking) tandis que le Version Space evite ce probleme en gardant les deux frontieres.

---

## 4. Algorithme Current-Best-Hypothesis (CBH)

L'algorithme CBH (AIMA Figure 19.2) maintient **une seule hypothese** et l'ajuste incrementalement :

1. Commencer avec la premiere hypothese consistante avec les exemples vus
2. Pour chaque nouvel exemple :
   - Si **positif** et non couvert -> **generaliser** l'hypothese
   - Si **negatif** et couvert -> **specialiser** l'hypothese
   - Si l'ajustement introduit des incoherences avec les exemples precedents -> **backtracking**

### Pseudo-code (AIMA 19.2)

```
function CBH(examples) returns hypothesis
    h <- premiere hypothese consistante avec examples[0]
    for each example in examples[1:]:
        if example est positif:
            if h ne couvre pas example:
                h <- generaliser(h, example)  // minimale
                if h inconsistante avec exemples precedents:
                    BACKTRACK (pas de solution simple)
        else:  // exemple negatif
            if h couvre example:
                h <- specialiser(h, example)  // minimale
                if h inconsistante avec exemples precedents:
                    BACKTRACK
    return h
```

In [4]:
def current_best_hypothesis(
    examples: list[dict],
    verbose: bool = True
) -> tuple[Optional[Hypothesis], list[dict]]:
    """Algorithme Current-Best-Hypothesis (AIMA Figure 19.2).
    
    Maintient une seule hypothese et l'ajuste incrementalement.
    Retourne (hypothese, trace) ou trace est la liste des etapes.
    """
    trace = []
    
    # Trouver la premiere hypothese consistante avec le premier exemple
    first = examples[0]
    if first["WillWait"]:
        # Pour un positif : l'hypothese exacte de l'exemple
        h = tuple(first[attr] for attr in ATTRIBUTES)
    else:
        # Pour un negatif : on ne peut pas initialiser avec S0 (trop specifique)
        # On utilise G0 et on laisse les exemples suivants specialiser
        h = G0
    
    seen = [first]
    trace.append({"step": 0, "example": 0, "label": first["WillWait"],
                   "action": "init", "hypothesis": h})
    
    for idx in range(1, len(examples)):
        ex = examples[idx]
        h_covers = covers(h, ex)
        action = "keep"
        
        if ex["WillWait"] and not h_covers:
            # Faux negatif -> generaliser, puis verifier que la generalisation
            # ne couvre pas un negatif deja vu (AIMA : sinon backtrack)
            h = generalize_for(h, ex)
            if is_consistent(h, seen + [ex]):
                action = "generalize"
            else:
                action = "generalize (INCONSISTANT)"
        elif not ex["WillWait"] and h_covers:
            # Faux positif -> specialiser
            # Essayer chaque specialisation et garder la premiere consistante
            specs = specialize_against(h, ex)
            found_valid = False
            for spec in specs:
                if is_consistent(spec, seen):
                    h = spec
                    found_valid = True
                    break
            if not found_valid:
                action = "BACKTRACK (no valid specialization)"
            else:
                action = "specialize"
        else:
            action = "keep (consistent)"
        
        seen.append(ex)
        trace.append({"step": idx, "example": idx, "label": ex["WillWait"],
                       "action": action, "hypothesis": h})
    
    return h, trace


# Execution de CBH sur les exemples du restaurant
cbh_result, cbh_trace = current_best_hypothesis(EXAMPLES, verbose=True)

print("Trace CBH :")
print(f"{'Step':>4} | {'Label':>5} | {'Action':<25} | Hypothese")
print("-" * 80)
for step in cbh_trace:
    label = "+" if step["label"] else "-"
    h_str = str(step["hypothesis"])[:50]
    print(f"{step['step']:>4} | {label:>5} | {step['action']:<25} | {h_str}")

print(f"\nHypothese finale CBH : {cbh_result}")
fp, fn = count_errors(cbh_result, EXAMPLES)
print(f"Faux positifs : {fp}, Faux negatifs : {fn}")
print(f"Consistante : {is_consistent(cbh_result, EXAMPLES)}")

Trace CBH :
Step | Label | Action                    | Hypothese
--------------------------------------------------------------------------------
   0 |     + | init                      | ('Yes', 'No', 'No', 'Yes', 'Some', '$$$', 'No', 'Y
   1 |     + | generalize                | ('Yes', 'No', 'No', 'Yes', '?', '?', 'No', '?', '?
   2 |     - | keep (consistent)         | ('Yes', 'No', 'No', 'Yes', '?', '?', 'No', '?', '?
   3 |     + | generalize                | ('Yes', 'No', '?', 'Yes', '?', '?', '?', '?', '?',
   4 |     - | keep (consistent)         | ('Yes', 'No', '?', 'Yes', '?', '?', '?', '?', '?',
   5 |     + | generalize                | ('?', '?', '?', 'Yes', '?', '?', '?', '?', '?', '?
   6 |     - | keep (consistent)         | ('?', '?', '?', 'Yes', '?', '?', '?', '?', '?', '?
   7 |     + | keep (consistent)         | ('?', '?', '?', 'Yes', '?', '?', '?', '?', '?', '?
   8 |     - | keep (consistent)         | ('?', '?', '?', 'Yes', '?', '?', '?', '?', '?', '?
   9 |  

### Interpretation : Resultat CBH

L'algorithme CBH ajuste progressivement l'hypothese a chaque nouvel exemple :

| Situation | Action | Effet |
|-----------|--------|-------|
| Exemple positif non couvert | Generaliser | Elargir l'hypothese pour couvrir |
| Exemple negatif couvert | Specialiser | Restreindre l'hypothese pour exclure |
| Deja consistent | Garder | Aucun changement |

**Limites du CBH** :
1. **Backtracking** : Quand aucune specialisation n'est consistante, il faut revenir en arriere (couteux)
2. **Ordre-dependance** : Le resultat depend de l'ordre des exemples
3. **Hypothese unique** : On ne garde qu'une seule hypothese, pas l'espace complet

> **Note** : CBH est un algorithme glouton qui peut echouer ou trouver un resultat sous-optimal selon l'ordre d'arrivee des exemples. Le Version Space resout ce probleme en gardant **toutes** les hypotheses consistantes.

---

## 5. Algorithme Version Space / Candidate Elimination

L'algorithme **Candidate Elimination** (AIMA Figure 19.3) resout le probleme du CBH en representant **tout l'espace des hypotheses consistantes** au moyen de deux frontieres :

- **G-set** (General boundary) : hypotheses les plus generales encore consistantes
- **S-set** (Specific boundary) : hypotheses les plus specifiques encore consistantes

Toute hypothese consistante se situe **entre** G et S : pour tout h consistant, il existe g dans G et s dans S tels que g >= h >= s.

### Algorithme

```
G <- {G0}  (hypothese la plus generale)
S <- {S0}  (hypothese la plus specifique)

for each example:
    if POSITIF:
        G <- retirer de G les hypotheses qui ne couvrent pas l'exemple
        Pour chaque s dans S qui ne couvre pas l'exemple:
            s <- generaliser(s) minimale pour couvrir l'exemple
            Retirer s si aucun element de G n'est plus general ou egal a s
        Retirer de S les hypotheses doublons ou non minimales
    if NEGATIF:
        S <- retirer de S les hypotheses qui couvrent l'exemple (inchange)
        Pour chaque g dans G qui couvre l'exemple:
            Remplacer g par ses specialisations qui ne couvrent pas l'exemple
            Retirer celles qui sont plus specifiques qu'un element de S
        Retirer de G les hypotheses doublons ou non maximales
```

In [5]:
def candidate_elimination(
    examples: list[dict],
    verbose: bool = True
) -> tuple[set[Hypothesis], set[Hypothesis], list[dict]]:
    """Algorithme Candidate Elimination (AIMA Figure 19.3).
    
    Retourne (G_set, S_set, trace).
    """
    G: set[Hypothesis] = {G0}
    S: set[Hypothesis] = {S0}
    trace = []
    
    for idx, ex in enumerate(examples):
        if ex["WillWait"]:
            # --- Exemple POSITIF ---
            # G : retirer les hypotheses qui ne couvrent pas l'exemple
            G = {g for g in G if covers(g, ex)}
            
            # S : generaliser chaque hypothese qui ne couvre pas l'exemple
            new_S: set[Hypothesis] = set()
            for s in S:
                if covers(s, ex):
                    new_S.add(s)
                else:
                    s_gen = generalize_for(s, ex)
                    new_S.add(s_gen)
            
            # Retirer de S les hypotheses plus generales qu'une autre
            S = _remove_subsumed(new_S)
            
            # Retirer de S les hypotheses pas plus generales qu'un element de G
            S = {s for s in S if any(is_more_general_than(g, s) for g in G)}
            
        else:
            # --- Exemple NEGATIF ---
            # S : retirer les hypotheses qui couvrent le negatif (elles sont inchangees)
            S = {s for s in S if not covers(s, ex)}
            
            # G : specialiser chaque hypothese qui couvre le negatif
            new_G: set[Hypothesis] = set()
            for g in G:
                if not covers(g, ex):
                    new_G.add(g)  # Gardee telle quelle
                else:
                    # Specialiser et ne garder que les specialisations consistantes
                    for spec in specialize_against(g, ex):
                        if not covers(spec, ex):  # ne couvre pas le negatif
                            # Verifier plus generale qu'un element de S
                            if any(is_more_general_than(spec, s) for s in S):
                                new_G.add(spec)
            
            # Retirer de G les hypotheses plus specifiques qu'une autre
            G = _remove_subsumed_G(new_G)
        
        trace.append({
            "example": idx,
            "label": ex["WillWait"],
            "|G|": len(G),
            "|S|": len(S),
        })
    
    return G, S, trace


def _remove_subsumed(S: set[Hypothesis]) -> set[Hypothesis]:
    """Retire de S les hypotheses qui sont plus generales qu'une autre dans S.
    On garde les hypotheses les PLUS specifiques.
    """
    result = set()
    s_list = list(S)
    for i, s1 in enumerate(s_list):
        subsumed = False
        for j, s2 in enumerate(s_list):
            if i != j and is_more_general_than(s1, s2):
                # s1 est plus generale que s2 -> s1 est subsumee
                subsumed = True
                break
        if not subsumed:
            result.add(s1)
    return result


def _remove_subsumed_G(G: set[Hypothesis]) -> set[Hypothesis]:
    """Retire de G les hypotheses qui sont plus specifiques qu'une autre dans G.
    On garde les hypotheses les PLUS generales.
    """
    result = set()
    g_list = list(G)
    for i, g1 in enumerate(g_list):
        subsumed = False
        for j, g2 in enumerate(g_list):
            if i != j and is_more_general_than(g2, g1):
                # g2 est plus generale que g1 -> g1 est subsumee
                subsumed = True
                break
        if not subsumed:
            result.add(g1)
    return result


# Execution
G_final, S_final, vs_trace = candidate_elimination(EXAMPLES)

print("Trace Version Space :")
print(f"{'Ex':>3} | {'Label':>5} | {'|G|':>3} | {'|S|':>3}")
print("-" * 25)
for step in vs_trace:
    label = "+" if step["label"] else "-"
    print(f"{step['example']:>3} | {label:>5} | {step['|G|']:>3} | {step['|S|']:>3}")

print(f"\nG-set final ({len(G_final)} hypotheses) :")
for g in sorted(G_final):
    print(f"  {g}")

print(f"\nS-set final ({len(S_final)} hypotheses) :")
for s in sorted(S_final):
    print(f"  {s}")

Trace Version Space :
 Ex | Label | |G| | |S|
-------------------------
  0 |     + |   1 |   1
  1 |     + |   1 |   1
  2 |     - |   3 |   1
  3 |     + |   3 |   1
  4 |     - |   1 |   1
  5 |     + |   1 |   1
  6 |     - |   1 |   1
  7 |     + |   1 |   1
  8 |     - |   1 |   1
  9 |     - |   0 |   0
 10 |     - |   0 |   0
 11 |     + |   0 |   0

G-set final (0 hypotheses) :

S-set final (0 hypotheses) :


### Interpretation : Resultat du Version Space

L'algorithme Candidate Elimination converge vers un ensemble d'hypotheses consistantes bornees par G et S :

| Ensemble | Role | Taille |
|----------|------|--------|
| **G-set** | Frontieres les plus generales | Variable |
| **S-set** | Frontieres les plus specifiques | Variable |
| **Version Space** | Toutes les hypotheses entre G et S | Grand mais implicite |

**Proprietes** :
- Toute hypothese h telle que `g >= h >= s` pour un g dans G et un s dans S est consistante
- Si G = S (frontieres identiques), une seule hypothese est consistante -> **convergence**
- Si G ou S devient vide, l'espace de version est vide -> pas de hypothese consistante

> **Point cle** : Le Version Space represente **implicitement** toutes les hypotheses consistantes. Meme s'il y en a des milliers, seules les frontieres G et S sont stockees.
> **Resultat observe ici** : sur les exemples du restaurant, G et S s'effondrent
> a l'exemple 9 (`|G| = |S| = 0`) : **aucune conjonction unique ne peut representer
> WillWait**. Le concept reel (AIMA 19.3) est un arbre de decision, c'est-a-dire
> une combinaison disjonctive de cas --- le biais de representation conjonctif est
> trop fort pour ce domaine. C'est un resultat pedagogique important, pas un bug :
> il motive les representations plus riches (arbres de decision, clauses multiples
> en SL-4). La section 6 contourne l'effondrement en apprenant le VS sur les
> 4 premiers exemples seulement --- la ou il est encore large.


---

## 6. Prediction avec le Version Space

Pour classifier un nouvel exemple, on peut utiliser 3 strategies :

1. **Unanime** : Predict positif si TOUTE hypothese dans VS couvre, negatif si AUCUNE ne couvre
2. **Majorite** : Vote majoritaire parmi les hypotheses du VS
3. **Conservateur** : Predict positif si S-set couvre, negatif si G-set ne couvre pas
> Le VS appris sur les 12 exemples etant **vide** (effondrement, section 5), on
> apprend ici les frontieres sur les **4 premiers exemples** et on evalue sur les
> 8 suivants, jamais vus (**test**). Pourquoi 4 et pas 9 ? A 9 exemples le VS a
> deja converge vers une hypothese unique : les trois strategies deviendraient
> identiques. A 4 exemples le VS est encore large (7 hypotheses) et les
> strategies divergent reellement --- c'est ce qu'on veut observer.


In [6]:
# Le VS sur les 12 exemples s'est effondre (section 5), et apres 9 exemples
# il a deja converge vers une hypothese unique (strategies identiques).
# Pour comparer les strategies, on se place la ou le VS est encore LARGE :
# apres les 4 premiers exemples (|G|=3, cf. trace section 5). Les 8 exemples
# suivants servent de test.
N_TRAIN = 4
G_pred, S_pred, _ = candidate_elimination(EXAMPLES[:N_TRAIN], verbose=False)
print(f"VS appris sur {N_TRAIN} exemples : |G|={len(G_pred)}, |S|={len(S_pred)}")


def enumerate_version_space(
    G: set[Hypothesis],
    S: set[Hypothesis],
    examples: list[dict]
) -> list[Hypothesis]:
    """Enumere toutes les hypotheses consistantes du Version Space.

    Astuce : toute hypothese consistante h satisfait h >= s pour un s du S-set.
    On genere donc, pour chaque s, les hypotheses obtenues en relachant a '?'
    un sous-ensemble de ses contraintes specifiees, puis on ne garde que
    celles qui restent consistantes avec tous les exemples.
    """
    from itertools import combinations
    vs: set[Hypothesis] = set()
    for s in S:
        specified = [i for i, v in enumerate(s) if v != "?"]
        for r in range(len(specified) + 1):
            for combo in combinations(specified, r):
                h = tuple("?" if i in combo else v for i, v in enumerate(s))
                if h not in vs and is_consistent(h, examples):
                    vs.add(h)
    return sorted(vs)


VS_ALL = enumerate_version_space(G_pred, S_pred, EXAMPLES[:N_TRAIN])
print(f"Version Space complet : {len(VS_ALL)} hypotheses consistantes")


def predict_vs(
    example: dict,
    G: set[Hypothesis],
    S: set[Hypothesis],
    strategy: str = "unanimous",
    vs_all: list[Hypothesis] = None
) -> tuple[bool, str]:
    """Prediction avec le Version Space.

    Retourne (prediction, justification).
    """
    s_covers = any(covers(s, example) for s in S)
    g_covers = any(covers(g, example) for g in G)

    if strategy == "unanimous":
        if s_covers and g_covers:
            return True, "Toutes les hypotheses couvrent"
        elif not s_covers and not g_covers:
            return False, "Aucune hypothese ne couvre"
        else:
            return None, "Desaccord entre hypotheses"  # type: ignore
    elif strategy == "majority":
        votes = sum(1 for h in vs_all if covers(h, example))
        total = len(vs_all)
        verdict = votes * 2 > total
        return verdict, f"Majorite : {votes}/{total} hypotheses couvrent"
    elif strategy == "conservative":
        if s_covers:
            return True, "S-set couvre (conservateur)"
        elif not g_covers:
            return False, "G-set ne couvre pas (conservateur)"
        else:
            return None, "Incertain (entre G et S)"  # type: ignore
    else:
        return None, f"Strategie inconnue: {strategy}"  # type: ignore


# Evaluation : 9 exemples d'entrainement + 3 exemples de test (jamais vus)
print()
print("Evaluation du Version Space (train = 0-3, TEST = 4-11) :")
print(f"{'Ex':>3} | {'Role':>5} | {'Vrai':>5} | {'Unanime':>8} | {'Majorite':>8} | {'Conserv.':>8} | Attributs cles")
print("-" * 86)

results = []
for i, ex in enumerate(EXAMPLES):
    role = "train" if i < N_TRAIN else "TEST"
    true_label = ex["WillWait"]

    pred_u, _ = predict_vs(ex, G_pred, S_pred, "unanimous")
    pred_m, _ = predict_vs(ex, G_pred, S_pred, "majority", VS_ALL)
    pred_c, _ = predict_vs(ex, G_pred, S_pred, "conservative")
    results.append((role, true_label, pred_u, pred_m, pred_c))

    fmt = lambda p: "+" if p is True else ("-" if p is False else "?")
    print(f"{i:>3} | {role:>5} | {'+' if true_label else '-':>5} | {fmt(pred_u):>8} |"
          f" {fmt(pred_m):>8} | {fmt(pred_c):>8} | Patrons={ex['Patrons']}, Type={ex['Type']}")

print()
for role in ("train", "TEST"):
    subset = [r for r in results if r[0] == role]
    n = len(subset)
    for name, k in [("Unanime", 2), ("Majorite", 3), ("Conservateur", 4)]:
        correct = sum(1 for r in subset if r[k] == r[1])
        uncertain = sum(1 for r in subset if r[k] is None)
        print(f"  {name:12s} ({role:5s}) : {correct}/{n} corrects, {uncertain} incertains")


VS appris sur 4 exemples : |G|=3, |S|=1
Version Space complet : 7 hypotheses consistantes

Evaluation du Version Space (train = 0-3, TEST = 4-11) :
 Ex |  Role |  Vrai |  Unanime | Majorite | Conserv. | Attributs cles
--------------------------------------------------------------------------------------
  0 | train |     + |        + |        + |        + | Patrons=Some, Type=French
  1 | train |     + |        + |        + |        + | Patrons=Full, Type=Thai
  2 | train |     - |        - |        - |        - | Patrons=Some, Type=Burger
  3 | train |     + |        + |        + |        + | Patrons=Full, Type=Thai
  4 |  TEST |     - |        ? |        - |        ? | Patrons=Full, Type=French
  5 |  TEST |     + |        ? |        - |        ? | Patrons=Some, Type=Italian
  6 |  TEST |     - |        - |        - |        - | Patrons=None, Type=Burger
  7 |  TEST |     + |        ? |        - |        ? | Patrons=Some, Type=Thai
  8 |  TEST |     - |        - |        - |        -

### Interpretation : Prediction

| Strategie | Principe | Resultat (test, 8 ex.) | Avantage | Inconvenient |
|-----------|----------|------------------------|----------|--------------|
| **Unanime** | Positif si tout le VS couvre, negatif si rien ne couvre | 2/8 corrects, **6 incertains** | Ne se prononce jamais a tort | Abstient des que le VS est large |
| **Majorite** | Vote des 7 hypotheses du VS | **5/8 corrects, 0 incertain** | Tranche toujours | Peut se tromper (ex. 5 et 7 : vrais positifs predits negatifs) |
| **Conservateur** | S-set couvre -> positif ; G-set ne couvre pas -> negatif | 2/8 corrects, 6 incertains | Bonne calibration | Memes abstentions que l'unanime ici |

**Observations** :

- Sur le **train**, les trois strategies font 4/4 : toute hypothese du VS est consistante avec ces exemples **par construction**.
- Sur le **test**, l'unanime et le conservateur ne se prononcent que sur les 2 negatifs flagrants (hors de portee meme de G) --- et ont raison. Les 6 autres exemples tombent dans la zone d'incertitude du VS.
- La **majorite** tranche toujours, mais penche vers le "non" : la plupart des 7 hypotheses restent proches du S-set (tres specifiques), donc couvrent peu. Elle rate ainsi les positifs 5 et 7.
- Le compromis est classique : **ne jamais se tromper** (unanime) contre **toujours repondre** (majorite). Plus on ajoute d'exemples, plus le VS se resserre et plus les strategies se rejoignent --- ici, des 9 exemples, le VS converge vers une hypothese unique et les trois strategies deviennent identiques.


---

## 7. Visualisation de l'evolution du Version Space

Observons comment les frontieres G et S evoluent au fil des exemples. L'ideal est une convergence rapide : G et S se rejoignent vers une hypothese unique.

In [7]:
# Evolution detaillee du Version Space
print("Evolution detaillee du Version Space :")
print("=" * 70)

G_run: set[Hypothesis] = {G0}
S_run: set[Hypothesis] = {S0}

for idx, ex in enumerate(EXAMPLES):
    label_str = "POSITIF" if ex["WillWait"] else "NEGATIF"
    attrs_short = f"Pat={ex['Patrons']}, Type={ex['Type']}, Price={ex['Price']}"
    
    # Sauvegarder l'etat avant
    g_before = len(G_run)
    s_before = len(S_run)
    
    # Mettre a jour
    if ex["WillWait"]:
        G_run = {g for g in G_run if covers(g, ex)}
        new_S = set()
        for s in S_run:
            if covers(s, ex):
                new_S.add(s)
            else:
                new_S.add(generalize_for(s, ex))
        S_run = _remove_subsumed(new_S)
        S_run = {s for s in S_run if any(is_more_general_than(g, s) for g in G_run)}
    else:
        S_run = {s for s in S_run if not covers(s, ex)}
        new_G = set()
        for g in G_run:
            if not covers(g, ex):
                new_G.add(g)
            else:
                for spec in specialize_against(g, ex):
                    if not covers(spec, ex):
                        if any(is_more_general_than(spec, s) for s in S_run):
                            new_G.add(spec)
        G_run = _remove_subsumed_G(new_G)
    
    delta_g = len(G_run) - g_before
    delta_s = len(S_run) - s_before
    
    print(f"\nEx {idx:>2} [{label_str:>7}] {attrs_short}")
    print(f"  G: {g_before} -> {len(G_run)} ({delta_g:+d})  |  S: {s_before} -> {len(S_run)} ({delta_s:+d})")
    
    if len(G_run) <= 5:
        for g in sorted(G_run):
            h_str = ", ".join(
                f"{ATTRIBUTES[i]}={g[i]}" for i in range(len(ATTRIBUTES)) if g[i] != "?"
            )
            print(f"    G: {h_str}")
    
    if len(S_run) <= 5:
        for s in sorted(S_run):
            h_str = ", ".join(
                f"{ATTRIBUTES[i]}={s[i]}" for i in range(len(ATTRIBUTES)) if s[i] is not None and s[i] != "?"
            )
            print(f"    S: {h_str}")
    
    if not G_run or not S_run:
        print("  ** VERSION SPACE VIDE **")
        break

Evolution detaillee du Version Space :

Ex  0 [POSITIF] Pat=Some, Type=French, Price=$$$
  G: 1 -> 1 (+0)  |  S: 1 -> 1 (+0)
    G: 
    S: Alternate=Yes, Bar=No, Fri/Sat=No, Hungry=Yes, Patrons=Some, Price=$$$, Raining=No, Reservation=Yes, Type=French, WaitEstimate=0-10

Ex  1 [POSITIF] Pat=Full, Type=Thai, Price=$
  G: 1 -> 1 (+0)  |  S: 1 -> 1 (+0)
    G: 
    S: Alternate=Yes, Bar=No, Fri/Sat=No, Hungry=Yes, Raining=No

Ex  2 [NEGATIF] Pat=Some, Type=Burger, Price=$
  G: 1 -> 3 (+2)  |  S: 1 -> 1 (+0)
    G: Hungry=Yes
    G: Bar=No
    G: Alternate=Yes
    S: Alternate=Yes, Bar=No, Fri/Sat=No, Hungry=Yes, Raining=No

Ex  3 [POSITIF] Pat=Full, Type=Thai, Price=$
  G: 3 -> 3 (+0)  |  S: 1 -> 1 (+0)
    G: Hungry=Yes
    G: Bar=No
    G: Alternate=Yes
    S: Alternate=Yes, Bar=No, Hungry=Yes

Ex  4 [NEGATIF] Pat=Full, Type=French, Price=$$$
  G: 3 -> 1 (-2)  |  S: 1 -> 1 (+0)
    G: Hungry=Yes
    S: Alternate=Yes, Bar=No, Hungry=Yes

Ex  5 [POSITIF] Pat=Some, Type=Italian, Price=$$


### Interpretation : Evolution du Version Space

L'evolution du Version Space suit un schema previsible :

| Phase | Comportement | Cause |
|-------|-------------|-------|
| Debut | G grandit, S se generalise | Les premiers exemples posent les frontieres |
| Milieu | G se specialise, S se generalise | Les frontieres convergent |
| Fin | Stabilisation ou convergence | L'espace se resserre autour de la solution |

**Indicateurs** :
- Si `|G|` et `|S|` restent petits : l'espace est bien contraint
- Si `|G|` explose : trop d'hypotheses generales possibles (besoin de plus d'exemples negatifs)
- Si le VS devient vide : les donnees sont bruitees ou l'hypothese n'est pas representable en conjonction pure

---

## 8. Limites du Version Space

Le Version Space a des limites fondamentales qui expliquent pourquoi il n'est pas utilise en pratique aujourd'hui.

### 8.1 Sensibilite au bruit

Un seul exemple mal etiquete peut vider le Version Space completement.

In [8]:
# Demonstration : ajout d'un exemple bruite
noisy_examples = EXAMPLES.copy()

# Creer un exemple bruite en inversant le label de l'exemple 0
noisy_ex = dict(EXAMPLES[0])
noisy_ex["WillWait"] = not noisy_ex["WillWait"]
noisy_examples.append(noisy_ex)

print(f"Exemple bruite ajoute : WillWait={noisy_ex['WillWait']} (inverse)")
print(f"Patrons={noisy_ex['Patrons']}, Type={noisy_ex['Type']}, Price={noisy_ex['Price']}")
print()

G_noisy, S_noisy, trace_noisy = candidate_elimination(noisy_examples)

if not G_noisy or not S_noisy:
    print("RESULTAT : Version Space VIDE !")
    print("Le bruit (un seul exemple mal etiquete) a detruit l'espace.")
else:
    print(f"G-set : {len(G_noisy)} hypotheses")
    print(f"S-set : {len(S_noisy)} hypotheses")

Exemple bruite ajoute : WillWait=False (inverse)


Patrons=Some, Type=French, Price=$$$

RESULTAT : Version Space VIDE !
Le bruit (un seul exemple mal etiquete) a detruit l'espace.


### 8.2 Incapacite a representer les disjonctions

Le Version Space ne represente que des **conjonctions**. Si le concept cible est une disjonction (ex: "Patrons=Some OU (Patrons=Full ET Fri/Sat=Yes)"), le VS ne peut pas le representer.

In [9]:
# Demonstration : concept disjonctif
# Le vrai concept AIMA est disjonctif :
# WillWait si (Patrons=Some) OU (Patrons=Full ET Hungry=Yes)

print("Concept cible AIMA (disjonctif) :")
print("  WillWait si Patrons=Some")
print("            OU (Patrons=Full AND Hungry=Yes)")
print()

# Verifions combien d'exemples satisfont chaque clause
clause1 = sum(1 for e in EXAMPLES if e["Patrons"] == "Some" and e["WillWait"])
clause2 = sum(1 for e in EXAMPLES if e["Patrons"] == "Full" and e["Hungry"] == "Yes" and e["WillWait"])
total_pos = sum(1 for e in EXAMPLES if e["WillWait"])

print(f"Clause 1 (Patrons=Some) couvre : {clause1} positifs")
print(f"Clause 2 (Patrons=Full AND Hungry=Yes) couvre : {clause2} positifs")
print(f"Total positifs : {total_pos}")
print()
print("Le Version Space ne peut pas representer cette disjonction.")
print("Il represente la meilleure approximation sous forme de conjonction,")
print("ce qui explique pourquoi il reste plusieurs hypotheses dans G et S.")
print()
print("Solutions possibles :")
print("  1. Arbres de decision (peuvent representer les disjonctions)")
print("  2. Apprentissage d'ensembles de regles (separate-and-conquer)")
print("  3. Modeles probabilistes (naive Bayes, regression logistique)")

Concept cible AIMA (disjonctif) :
  WillWait si Patrons=Some
            OU (Patrons=Full AND Hungry=Yes)

Clause 1 (Patrons=Some) couvre : 3 positifs
Clause 2 (Patrons=Full AND Hungry=Yes) couvre : 3 positifs
Total positifs : 6

Le Version Space ne peut pas representer cette disjonction.
Il represente la meilleure approximation sous forme de conjonction,
ce qui explique pourquoi il reste plusieurs hypotheses dans G et S.

Solutions possibles :
  1. Arbres de decision (peuvent representer les disjonctions)
  2. Apprentissage d'ensembles de regles (separate-and-conquer)
  3. Modeles probabilistes (naive Bayes, regression logistique)


### Interpretation : Limites du Version Space

| Limite | Cause | Consequence | Solution alternative |
|--------|-------|-------------|---------------------|
| **Bruit** | Un seul exemple erronee | VS vide | Modeles probabilistes (tolerance aux erreurs) |
| **Disjonctions** | Reprsentation conjonctive seule | Approximation sous-optimale | Arbres de decision, ensembles de regles |
| **Explosion combinatoire** | Nombre de specialisations | G-set tres grand | Heuristiques, attribution de probabilites |
| **Attributs continus** | Besoin de seuils discrets | Non applicable directement | Arbres de decision avec seuils |

> **Note historique** : Malgre ses limites, le Version Space est fondamental en apprentissage symbolique. Il a inspire des methodes plus robustes comme les arbres de decision (ID3/C4.5) et les modeles de theorie des versions probabilistes.

---

## 9. Exercices

Les **exemples guides** ci-dessous sont resolus et commentes ; les **exercices** sont a completer par vos soins. Chaque exercice reutilise les implementations du notebook (`current_best_hypothesis`, `candidate_elimination`, `count_errors`).


### Exemple guide : CBH avec un ordre different

L'ordre des exemples influence le resultat de CBH. Voici ce qui se passe quand on place les negatifs en premier.

In [10]:
# Exemple guide : CBH avec un ordre different (negatifs d'abord)
positifs = [e for e in EXAMPLES if e["WillWait"]]
negatifs = [e for e in EXAMPLES if not e["WillWait"]]

reordered = negatifs + positifs

cbh_reord, trace_reord = current_best_hypothesis(reordered)

print(f"Hypothese CBH (negatifs d'abord) : {cbh_reord}")
fp_r, fn_r = count_errors(cbh_reord, EXAMPLES)
print(f"Faux positifs : {fp_r}, Faux negatifs : {fn_r}")
print(f"Consistante : {is_consistent(cbh_reord, EXAMPLES)}")

Hypothese CBH (negatifs d'abord) : ('?', '?', '?', 'Yes', '?', '?', '?', '?', '?', '?')
Faux positifs : 1, Faux negatifs : 0
Consistante : False


### Interpretation : CBH avec un ordre different

Le resultat obtenu avec les negatifs en premier est **identique** a celui obtenu avec l'ordre original : l'hypothese `Hungry=Yes` avec 1 faux positif.

| Aspect | Observation |
|--------|-------------|
| Hypothese finale | `('?', '?', '?', 'Yes', '?', ...)` (Hungry=Yes) |
| Consistance | Non (1 faux positif) |
| Convergence | Le backtracking a l'exemple 9 n'a pas pu etre resolu |

> **Note** : Dans ce cas precis, l'ordre n'a pas change le resultat final. Cependant, pour d'autres jeux de donnees ou d'autres ordres, CBH peut produire des hypotheses differentes. C'est cette instabilite qui motive l'utilisation du Version Space.

### Exercice 1 : CBH avec ordre personnalise

Testez CBH avec un ordre **aleatoire** des exemples. Executez plusieurs fois et comparez les hypotheses obtenues. Que constatez-vous sur la stabilite de CBH ?

**Etapes** :
1. Melangez les exemples avec `import random; random.shuffle(shuffled)`
2. Executez CBH sur les exemples melanges
3. Repetez 3 fois et comparez les resultats

In [11]:
import random

# TODO etudiant : testez CBH avec un ordre aleatoire (3 executions)
# Indice : copiez EXAMPLES, melangez avec random.shuffle, executez current_best_hypothesis
random.seed(42)
shuffled = EXAMPLES.copy()
random.shuffle(shuffled)
print("Exercice a completer : testez CBH avec un ordre aleatoire")
print("Comparez les hypotheses finales sur 3 executions avec des seeds differents")

Exercice a completer : testez CBH avec un ordre aleatoire
Comparez les hypotheses finales sur 3 executions avec des seeds differents


### Exercice 2 : Version Space sur un sous-ensemble

Executez le Candidate Elimination sur les 6 premiers exemples seulement et observez la taille du Version Space.

In [12]:
# Exercice 2 : Version Space sur un sous-ensemble
# TODO etudiant : Executez candidate_elimination sur EXAMPLES[:6]
# Etape 1 : Executer sur les 6 premiers exemples
result = None  # TODO etudiant : remplacer par l'appel a candidate_elimination(EXAMPLES[:6])

# Etape 2 : Afficher les resultats
if result is not None:
    G_sub, S_sub, trace_sub = result
    print(f"G-set ({len(G_sub)} hypotheses) :")
    for g in sorted(G_sub):
        print(f"  {g}")
    print(f"\nS-set ({len(S_sub)} hypotheses) :")
    for s in sorted(S_sub):
        print(f"  {s}")
else:
    print("Exercice a completer : appelez candidate_elimination(EXAMPLES[:6])")

# Indice : Comparez |G| et |S| avec le resultat sur les 12 exemples.
# Le Version Space est-il plus grand ou plus petit avec moins d'exemples ?

Exercice a completer : appelez candidate_elimination(EXAMPLES[:6])


### Exercice 3 : Apprentissage de regles avec couverture

Implementer un algorithme de couverture sequentiel pour apprendre des regles logiques.

**Indice** : Pour chaque regle, specialisez jusqu'a couvrir uniquement des positifs.


In [13]:
# Exercice : Apprentissage de regles avec couverture
# TODO etudiant : Implementer un algorithme de couverture sequentiel pour apprendre des regles logiques
# Indice : Pour chaque regle, specialisez jusqu'a couvrir uniquement des positifs
result = None  # TODO etudiant : remplacer par votre implementation
print("Exercice a completer : Apprentissage de regles avec couverture")


Exercice a completer : Apprentissage de regles avec couverture


### Exemple guide : Reflexion corrigee (synthese)

Pour conclure, voici une demonstration de la demarche attendue sur trois questions de synthese, avec leur correction. Observez la structure des reponses : un **mecanisme precis** (pas une generalite), puis sa **consequence pratique**. L'exercice 4 ci-dessous vous propose trois questions du meme type --- mais distinctes --- a traiter de la meme maniere.

| Question | Reponse |
|----------|--------|
| Pourquoi CBH depend de l'ordre des exemples, mais pas le Version Space ? | Le VS maintient TOUTES les hypotheses consistantes, pas une seule. L'ordre n'affecte que la vitesse de convergence, pas le resultat final. |
| Pourquoi la prediction par vote majoritaire (section 6) peut-elle etre utile alors meme que le Version Space n'a pas converge ? | Chaque hypothese consistante "vote" sur le nouvel exemple : si une large majorite predit la meme classe, la prediction est fiable sans attendre la convergence. Seuls les cas partages (vote proche de 50/50) doivent etre refuses ou renvoyes a l'acquisition de nouveaux exemples. |
| Pourquoi le Version Space est-il fragile face au bruit ? | Un seul exemple contradictoire elimine TOUTES les hypotheses, vidant le VS. Pas de mecanisme de tolerance. |


### Exercice 4 : Reflexion

A votre tour, sur le modele de l'exemple guide ci-dessus. Repondez aux questions suivantes en vous appuyant sur les algorithmes implementes plus haut (CBH, Candidate Elimination) : pour chaque reponse, identifiez le mecanisme en jeu, puis sa consequence pratique.

| Question | Reponse |
|----------|--------|
| Donnez une sequence concrete d'exemples (positifs/negatifs) qui force CBH a backtracker, alors que le Version Space converge sans jamais revenir en arriere. | _(a completer)_ |
| Un concept disjonctif comme "animal = chien OU chat" n'est pas capturable par une hypothese conjonctive unique. Pourquoi ? Que faudrait-il changer dans la representation des hypotheses pour le permettre ? | _(a completer)_ |
| La complexite memoire du Version Space est O(\|G\| + \|S\|). Decrivez un cas ou \|G\| ou \|S\| explose, et la consequence pratique sur l'apprentissage. | _(a completer)_ |


---

## 10. Resume

### Points cles

| Concept | Description |
|---------|-------------|
| **Hypothese FOL** | Conjonction de contraintes sur les attributs |
| **Consistance** | Pas de faux positifs ni de faux negatifs |
| **Generalisation** | Remplacer une contrainte par `?` (couvrir plus) |
| **Specialisation** | Remplacer `?` par une valeur (couvrir moins) |
| **CBH** | Maintient une seule hypothese, ajustee incrementalement |
| **Version Space** | Maintient G-set et S-set, represente toutes les hypotheses consistantes |
| **Candidate Elimination** | Algorithme pour mettre a jour G et S efficacement |

### Comparaison CBH vs Version Space

| Critere | CBH | Version Space |
|---------|-----|---------------|
| Nombre d'hypotheses | 1 | Toutes (via frontieres) |
| Ordre-dependance | Oui | Non |
| Backtracking | Possible | Jamais |
| Bruit | Robuste (peut revenir) | Fragile (VS vide) |
| Complexite memoire | O(1) | O(|G| + |S|) |


### Perspectives

Ce notebook a explore les methodes d'apprentissage logique symbolique dans le cadre de la contrainte d'entrainement : trouver une hypothese H telle que H et les descriptions entrainent logiquement les classifications. L'algorithme CBH (Current-Best-Hypothesis) maintient une unique hypothese ajustee incrementalement par generalisation et specialisation, avec un risque de backtracking lorsque aucune specialisation consistante n'existe. L'algorithme Candidate Elimination resout ce probleme en representant l'ensemble complet des hypotheses consistantes via les frontieres G-set et S-set, garantissant qu'aucune hypothese valide n'est oubliee. Cependant, les deux approches partagent des limites fondamentales : la fragilite face au bruit (un seul exemple mal etiquete vide le Version Space) et l'incapacite a representer des concepts disjonctifs avec des conjonctions pures.

Ces limites ont historiquement motive le passage a des modeles plus robustes : arbres de decision (ID3/C4.5) pour les disjonctions, modeles probabilistes (naive Bayes, regression logistique) pour la tolerance au bruit, et plus recemment les reseaux de neurones pour l'apprentissage de representations continues. Neanmoins, les concepts de generalisation, specialisation et espace de versions restent fondamentaux en apprentissage automatique symbolique et informent les methodes avancees comme la programmation logique inductive (ILP) et l'apprentissage neuro-symbolique.

Le notebook suivant, [SL-2 - Apprentissage et Connaissance](SL-2-KnowledgeBasedLearning.ipynb), depasse le cadre inductif pur en integrant la connaissance prealable du domaine. Nous y decouvrirons l'apprentissage base sur les explications (EBL), qui extrait une regle generale d'un seul exemple par deduction, et l'apprentissage base sur la pertinence (RBL), qui reduit exponentiellement l'espace des hypotheses grace aux determinations.

---

## Ressources

- Russell & Norvig, *AI: A Modern Approach*, 3e ed., Chapitre 19.1
- Tom Mitchell, *Machine Learning*, Chapitre 2 (Concept Learning)
- [AIMA Python Code](https://github.com/aimacode/aima-python) - Implementations de reference
- [Version Space - Wikipedia](https://en.wikipedia.org/wiki/Version_space)

---

**Notebook suivant** : [SL-2 - Apprentissage et Connaissance](SL-2-KnowledgeBasedLearning.ipynb)